In [3]:
from pathlib import Path
import gc

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score, classification_report

FEATURE_ROOT = Path(r"D:\LOSO_MEDIAPIPE")
FEATURE_DIR = FEATURE_ROOT / "features"
METADATA_PATH = FEATURE_ROOT / "metadata.csv"
MODEL_DIR = FEATURE_ROOT / "loso_models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

assert FEATURE_DIR.exists(), f"Features folder not found: {FEATURE_DIR}"
assert METADATA_PATH.exists(), f"Metadata file not found: {METADATA_PATH}"

NUM_FRAMES = 30
NUM_FEATURES = 225
BATCH_SIZE = 32
EPOCHS = 80
SEED = 42

tf.keras.utils.set_random_seed(SEED)

metadata = pd.read_csv(METADATA_PATH)
metadata["label"] = metadata["label"].astype(int)

print("Total samples:", len(metadata))
print("Signers:", sorted(metadata["signer"].unique()))
print("Classes:", metadata["label"].nunique())

Total samples: 3200
Signers: ['S1', 'S2', 'S3', 'S4']
Classes: 50


In [8]:
# Cell 2 — Feature loading with sequence-length correction
def resample_to_30_frames(landmarks: np.ndarray) -> np.ndarray:
    """
    Convert any valid (T, 225) landmark sequence to exactly (30, 225)
    using uniform temporal sampling. Short sequences repeat frames.
    """
    if landmarks.ndim != 2 or landmarks.shape[1] != NUM_FEATURES:
        raise ValueError(
            f"Expected feature shape (T, {NUM_FEATURES}), "
            f"but received {landmarks.shape}"
        )

    original_frames = landmarks.shape[0]

    if original_frames == 0:
        return np.zeros((NUM_FRAMES, NUM_FEATURES), dtype=np.float32)

    if original_frames == NUM_FRAMES:
        return landmarks.astype(np.float32)

    indices = np.linspace(
        0,
        original_frames - 1,
        num=NUM_FRAMES
    ).round().astype(np.int32)

    return landmarks[indices].astype(np.float32)


def load_features(dataframe: pd.DataFrame):
    """Load NPZ feature sequences as consistent (30, 225) float32 arrays."""
    sequences = []
    labels = []

    for row in dataframe.itertuples(index=False):
        feature_path = FEATURE_DIR / row.feature_file

        if not feature_path.exists():
            raise FileNotFoundError(f"Feature file not found: {feature_path}")

        with np.load(feature_path) as archive:
            if "landmarks" not in archive:
                raise KeyError(f"'landmarks' key is missing in {feature_path}")

            landmarks = archive["landmarks"]

        landmarks = resample_to_30_frames(landmarks)

        sequences.append(landmarks)
        labels.append(int(row.label))

    X = np.stack(sequences).astype(np.float32)
    y = np.asarray(labels, dtype=np.int32)

    assert X.shape[1:] == (NUM_FRAMES, NUM_FEATURES), X.shape

    return X, y

In [9]:
# Cell 3 — Train-only feature standardization
def standardize_train_test(X_train, X_test):
    """
    Compute mean/std from training signers only.
    This prevents test-signer information leakage.
    """
    mean = X_train.reshape(-1, NUM_FEATURES).mean(axis=0)
    std = X_train.reshape(-1, NUM_FEATURES).std(axis=0)

    std[std < 1e-6] = 1.0

    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std

    return X_train.astype(np.float32), X_test.astype(np.float32)

In [10]:
# Cell 4 — BiLSTM sign-recognition model
def build_model(num_classes: int):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(NUM_FRAMES, NUM_FEATURES)),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(128, return_sequences=True, dropout=0.30)
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64, dropout=0.30)
        ),

        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.40),

        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [11]:
# Cell 5 — Leave-One-Signer-Out training
num_classes = metadata["label"].nunique()
signers = sorted(metadata["signer"].unique())
fold_results = []

for test_signer in signers:
    print("\n" + "=" * 70)
    print(f"LOSO Fold — Test signer: {test_signer}")
    print("=" * 70)

    train_df = metadata[metadata["signer"] != test_signer].reset_index(drop=True)
    test_df = metadata[metadata["signer"] == test_signer].reset_index(drop=True)

    X_train, y_train = load_features(train_df)
    X_test, y_test = load_features(test_df)

    X_train, X_test = standardize_train_test(X_train, X_test)

    print("Training shape:", X_train.shape)
    print("Testing shape: ", X_test.shape)

    model = build_model(num_classes)

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=12,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            patience=5,
            factor=0.5,
            min_lr=1e-6,
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(MODEL_DIR / f"best_model_test_{test_signer}.keras"),
            monitor="val_accuracy",
            save_best_only=True,
        ),
    ]

    history = model.fit(
        X_train,
        y_train,
        validation_split=0.10,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=True,
        callbacks=callbacks,
        verbose=1,
    )

    probabilities = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
    y_pred = np.argmax(probabilities, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

    print(f"\nTest signer: {test_signer}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")

    print(classification_report(
        y_test,
        y_pred,
        digits=4,
        zero_division=0,
    ))

    fold_results.append({
        "test_signer": test_signer,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
    })

    del X_train, X_test, y_train, y_test, model
    tf.keras.backend.clear_session()
    gc.collect()


LOSO Fold — Test signer: S1
Training shape: (2400, 30, 225)
Testing shape:  (800, 30, 225)
Epoch 1/80
68/68 ━━━━━━━━━━━━━━━━━━━━ 31s 163ms/step - accuracy: 0.0662 - loss: 3.7067 - val_accuracy: 0.0000e+00 - val_loss: 3.9311 - learning_rate: 0.0010
Epoch 2/80
68/68 ━━━━━━━━━━━━━━━━━━━━ 17s 111ms/step - accuracy: 0.2417 - loss: 2.6567 - val_accuracy: 0.0167 - val_loss: 2.9935 - learning_rate: 0.0010
Epoch 3/80
68/68 ━━━━━━━━━━━━━━━━━━━━ 7s 109ms/step - accuracy: 0.4468 - loss: 1.7444 - val_accuracy: 0.1208 - val_loss: 2.8403 - learning_rate: 0.0010
Epoch 4/80
68/68 ━━━━━━━━━━━━━━━━━━━━ 10s 106ms/step - accuracy: 0.6037 - loss: 1.1848 - val_accuracy: 0.1208 - val_loss: 2.8791 - learning_rate: 0.0010
Epoch 5/80
68/68 ━━━━━━━━━━━━━━━━━━━━ 11s 115ms/step - accuracy: 0.7079 - loss: 0.8683 - val_accuracy: 0.1500 - val_loss: 3.6183 - learning_rate: 0.0010
Epoch 6/80
68/68 ━━━━━━━━━━━━━━━━━━━━ 8s 111ms/step - accuracy: 0.7681 - loss: 0.6734 - val_accuracy: 0.1875 - val_loss: 3.0424 - learning_r

In [12]:
# Cell 6 — Final LOSO result
results_df = pd.DataFrame(fold_results)

print("\nLOSO results per signer:")
print(results_df.to_string(index=False))

print("\nFinal signer-independent result:")
print(f"Mean Accuracy: {results_df['accuracy'].mean():.4f} "
      f"± {results_df['accuracy'].std(ddof=1):.4f}")

print(f"Mean Macro F1: {results_df['macro_f1'].mean():.4f} "
      f"± {results_df['macro_f1'].std(ddof=1):.4f}")

results_df.to_csv(FEATURE_ROOT / "loso_results.csv", index=False)


LOSO results per signer:
test_signer  accuracy  macro_f1
         S1   0.20500  0.150021
         S2   0.30500  0.259527
         S3   0.21250  0.144647
         S4   0.33125  0.271287

Final signer-independent result:
Mean Accuracy: 0.2634 ± 0.0641
Mean Macro F1: 0.2064 ± 0.0684
